# Aurora Bay FAQ — RAG on BigQuery

A retrieval-augmented chatbot for the fictional town of Aurora Bay, Alaska, built on BigQuery.

Steps:
1. Load the FAQ CSV from Cloud Storage into a BigQuery table.
2. Create a Vertex AI remote connection and an embedding model, then embed each question-and-answer pair and store the vectors alongside the original fields.
3. Answer a question: embed it, run `VECTOR_SEARCH` to retrieve the most relevant FAQs, then pass those plus the question to Gemini for a grounded answer.

Source file: `gs://labs.roitraining.com/aurora-bay-faqs/aurora-bay-faqs.csv`

## Setup

In [12]:
!pip install -q google-cloud-bigquery google-cloud-bigquery-connection db-dtypes pandas

## Configuration

The dataset and connection use the `US` multi-region. Update `PROJECT_ID` to match your lab project.

In [13]:
PROJECT_ID = "qwiklabs-gcp-00-fc2622edeeb6"
LOCATION = "US"
DATASET = "aurora_bay"

CONNECTION_ID = "vertex_conn"
EMBEDDING_MODEL = "embedding_model"
GEMINI_MODEL = "gemini_model"

# Vertex AI endpoints the remote models call
EMBEDDING_ENDPOINT = "text-embedding-005"
GEMINI_ENDPOINT = "gemini-2.5-flash"

# Source data
SOURCE_URI = "gs://labs.roitraining.com/aurora-bay-faqs/aurora-bay-faqs.csv"

DS = f"{PROJECT_ID}.{DATASET}"
CONNECTION_PATH = f"{PROJECT_ID}.{LOCATION}.{CONNECTION_ID}"

## BigQuery client

In [14]:
from google.cloud import bigquery
import time

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

def run(sql, params=None):
    """Run a query; return a dataframe, or None for statements with no rows."""
    config = bigquery.QueryJobConfig(query_parameters=params or [])
    job = client.query(sql, job_config=config)
    job.result()
    try:
        return job.to_dataframe()
    except Exception:
        return None

# Create the dataset if it doesn't already exist
dataset = bigquery.Dataset(DS)
dataset.location = LOCATION
client.create_dataset(dataset, exists_ok=True)
print("Dataset ready:", DS)

Dataset ready: qwiklabs-gcp-00-fc2622edeeb6.aurora_bay


## Load the CSV from Cloud Storage

Load directly from the GCS URI with schema auto-detection. The first row is treated as the header.

In [15]:
load_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition="WRITE_TRUNCATE",
)

faqs_table = f"{DS}.faqs"
client.load_table_from_uri(SOURCE_URI, faqs_table, job_config=load_config).result()

table = client.get_table(faqs_table)
print(f"Loaded {table.num_rows} rows into {faqs_table}")
print("Columns:", [(f.name, f.field_type) for f in table.schema])

# Preview a few rows
run(f"SELECT * FROM `{faqs_table}` LIMIT 5")

Loaded 50 rows into qwiklabs-gcp-00-fc2622edeeb6.aurora_bay.faqs
Columns: [('string_field_0', 'STRING'), ('string_field_1', 'STRING')]


,string_field_0,string_field_1
0,When was Aurora Bay founded?,Aurora Bay was founded in 1901 by a group of f...
1,What is the population of Aurora Bay?,Aurora Bay has a population of approximately 3...
2,Where is the Aurora Bay Town Hall located?,The Town Hall is located at 100 Harbor View Ro...
3,Who is the current mayor of Aurora Bay?,"The current mayor is Linda Greenwood, elected ..."
4,What are the primary industries in Aurora Bay?,The primary industries include commercial fish...


## Build the text to embed

We detect the question and answer columns from the loaded schema (case-insensitive). If both are found we embed `"question answer"`; otherwise we fall back to concatenating every text column. Building this expression from the detected names keeps the notebook working regardless of the exact column casing in the file.

In [16]:
columns = [f.name for f in table.schema]
string_columns = [f.name for f in table.schema if f.field_type == "STRING"]

q_col = next((c for c in columns if "question" in c.lower()), None)
a_col = next((c for c in columns if "answer" in c.lower()), None)

if q_col and a_col:
    content_expr = f"CONCAT({q_col}, ' ', {a_col})"
else:
    # Fallback: concatenate all string columns
    content_expr = "CONCAT(" + ", ' ', ".join(string_columns) + ")"

print("Question column:", q_col)
print("Answer column:", a_col)
print("Content expression:", content_expr)

Question column: None
Answer column: None
Content expression: CONCAT(string_field_0, ' ', string_field_1)


## Remote connection

BigQuery calls Vertex AI through a Cloud Resource connection. This cell creates the connection (or reuses it if it already exists), grants its service account the Vertex AI User role, and pauses for IAM propagation.

If the lab already provided a connection, set `CONNECTION_ID` above to match it and skip this cell.

In [17]:
from google.cloud import bigquery_connection_v1 as bqc

conn_client = bqc.ConnectionServiceClient()
parent = f"projects/{PROJECT_ID}/locations/{LOCATION.lower()}"

try:
    connection = bqc.Connection(cloud_resource=bqc.CloudResourceProperties())
    created = conn_client.create_connection(
        parent=parent, connection_id=CONNECTION_ID, connection=connection
    )
    service_account = created.cloud_resource.service_account_id
    print("Created connection.")
except Exception:
    name = f"{parent}/connections/{CONNECTION_ID}"
    existing = conn_client.get_connection(name=name)
    service_account = existing.cloud_resource.service_account_id
    print("Connection already exists.")

print("Service account:", service_account)

!gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member="serviceAccount:{service_account}" \
    --role="roles/aiplatform.user" --quiet --condition=None

print("Waiting 60s for IAM to propagate...")
time.sleep(90)

Connection already exists.
Service account: bqcx-824265081161-s12q@gcp-sa-bigquery-condel.iam.gserviceaccount.com
Updated IAM policy for project [qwiklabs-gcp-00-fc2622edeeb6].
bindings:
- members:
  - serviceAccount:service-824265081161@gcp-sa-vertex-nb.iam.gserviceaccount.com
  role: roles/aiplatform.colabServiceAgent
- members:
  - serviceAccount:service-824265081161@gcp-sa-aiplatform-vm.iam.gserviceaccount.com
  role: roles/aiplatform.notebookServiceAgent
- members:
  - serviceAccount:service-824265081161@gcp-sa-aiplatform.iam.gserviceaccount.com
  role: roles/aiplatform.serviceAgent
- members:
  - serviceAccount:bqcx-824265081161-s12q@gcp-sa-bigquery-condel.iam.gserviceaccount.com
  role: roles/aiplatform.user
- members:
  - serviceAccount:qwiklabs-gcp-00-fc2622edeeb6@qwiklabs-gcp-00-fc2622edeeb6.iam.gserviceaccount.com
  role: roles/bigquery.admin
- members:
  - serviceAccount:824265081161@cloudbuild.gserviceaccount.com
  role: roles/cloudbuild.builds.builder
- members:
  - servi

## Embedding model

In [18]:
run(f"""
CREATE OR REPLACE MODEL `{DS}.{EMBEDDING_MODEL}`
REMOTE WITH CONNECTION `{CONNECTION_PATH}`
OPTIONS (ENDPOINT = '{EMBEDDING_ENDPOINT}')
""")
print("Embedding model created.")

Embedding model created.


## Generate and store embeddings

`ML.GENERATE_EMBEDDING` embeds the content for every FAQ. We select all original columns plus the built `content`, so the resulting `faq_embeddings` table keeps every existing field next to the embedding vector. The `RETRIEVAL_DOCUMENT` task type marks these as documents to be retrieved later.

In [19]:
run(f"""
CREATE OR REPLACE TABLE `{DS}.faq_embeddings` AS
SELECT *
FROM ML.GENERATE_EMBEDDING(
  MODEL `{DS}.{EMBEDDING_MODEL}`,
  (SELECT *, {content_expr} AS content FROM `{DS}.faqs`),
  STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_DOCUMENT' AS task_type)
)
""")

# An empty status string means the embedding succeeded
run(f"""
SELECT COUNTIF(ml_generate_embedding_status != '') AS errors,
       COUNT(*) AS total
FROM `{DS}.faq_embeddings`
""")

,errors,total
0,0,50


## Gemini model

In [20]:
run(f"""
CREATE OR REPLACE MODEL `{DS}.{GEMINI_MODEL}`
REMOTE WITH CONNECTION `{CONNECTION_PATH}`
OPTIONS (ENDPOINT = '{GEMINI_ENDPOINT}')
""")
print("Gemini model created.")

Gemini model created.


## Vector search

Embed the user's question with the same model (task type `RETRIEVAL_QUERY`) and return the closest FAQs by cosine distance.

In [21]:
def search(question, top_k=3):
    sql = f"""
    SELECT base.content, distance
    FROM VECTOR_SEARCH(
      TABLE `{DS}.faq_embeddings`,
      'ml_generate_embedding_result',
      (
        SELECT ml_generate_embedding_result
        FROM ML.GENERATE_EMBEDDING(
          MODEL `{DS}.{EMBEDDING_MODEL}`,
          (SELECT @q AS content),
          STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_QUERY' AS task_type)
        )
      ),
      top_k => {top_k},
      distance_type => 'COSINE'
    )
    ORDER BY distance
    """
    params = [bigquery.ScalarQueryParameter("q", "STRING", question)]
    return run(sql, params)

search("What time does the town hall open?")

,content,distance
0,When are the town council meetings held? Town ...,0.312837
1,Where is the Aurora Bay Town Hall located? The...,0.369541
2,How can I volunteer in community events? You c...,0.421218


## RAG answer

One query does retrieval and generation together: `VECTOR_SEARCH` pulls the top FAQs, they're concatenated into a context block, and `ML.GENERATE_TEXT` has Gemini answer using only that context.

In [24]:
def answer(question, top_k=3):
    sql = f"""
    WITH retrieved AS (
      SELECT base.content AS content, distance
      FROM VECTOR_SEARCH(
        TABLE `{DS}.faq_embeddings`,
        'ml_generate_embedding_result',
        (
          SELECT ml_generate_embedding_result
          FROM ML.GENERATE_EMBEDDING(
            MODEL `{DS}.{EMBEDDING_MODEL}`,
            (SELECT @q AS content),
            STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_QUERY' AS task_type)
          )
        ),
        top_k => {top_k},
        distance_type => 'COSINE'
      )
      ORDER BY distance
    ),
    context AS (
      SELECT STRING_AGG(content, '\\n\\n') AS facts FROM retrieved
    )
    SELECT ml_generate_text_llm_result AS response
    FROM ML.GENERATE_TEXT(
      MODEL `{DS}.{GEMINI_MODEL}`,
      (
        SELECT CONCAT(
          'You are the Aurora Bay information assistant. ',
          'Answer the question using only the information below. ',
          'If it is not covered, say you do not have that information.\\n\\n',
          'Information:\\n', facts,
          '\\n\\nQuestion: ', @q, '\\nAnswer:'
        ) AS prompt
        FROM context
      ),
      STRUCT(TRUE AS flatten_json_output, 0.2 AS temperature, 512 AS max_output_tokens)
    )
    """
    params = [bigquery.ScalarQueryParameter("q", "STRING", question)]
    result = run(sql, params)
    return result["response"].iloc[0]

print(answer("When can I watch the northern lights?"))

You can watch the northern lights from October through March.


## Try a few questions

In [25]:
for q in [
    "How do I pay my water bill?",
    "When does the ferry run?",
    "Do I need a permit to build a deck?",
]:
    print("Q:", q)
    print("A:", answer(q))
    print("-" * 60)

Q: How do I pay my water bill?
A: You can pay water and sewer bills online, by mail, or in person at the Aurora Bay Utilities Department located within Town Hall.
------------------------------------------------------------
Q: When does the ferry run?
A: I do not have that information.
------------------------------------------------------------
Q: Do I need a permit to build a deck?
A: You do not have that information.
------------------------------------------------------------


## Interactive chat

Run this cell to chat with the assistant. Submit an empty line to stop.

In [26]:
while True:
    q = input("You: ").strip()
    if not q:
        break
    print("Aurora Bay Assistant:", answer(q))
    print()

You: Where  is Aurora Bay?
Aurora Bay Assistant: I do not have that information.

You: is in Alaska
Aurora Bay Assistant: You do not have that information.

You: When was Aurora Bay founded 
Aurora Bay Assistant: Aurora Bay was founded in 1901 by a group of fur traders who recognized the region’s strategic coastal location.

You: what type of fur was exported 
Aurora Bay Assistant: I do not have that information.

You: Where is the Aurora Bay Town Hall located?
Aurora Bay Assistant: The Town Hall is located at 100 Harbor View Road, in the center of Aurora Bay, close to the main harbor.

You: 
